# Graph Conformal Prediction with Any2Graph & the Coloring Dataset

Load the coloring dataset.

Generate it with the following command:

```bash
cd Models/Any2Graph/Any2Graph/Img2Graph/Coloring
python Coloring_Generate_Data.py --name small --train_size 100000 --Mmin 5 --Mmax 10
```

In [1]:
from Any2Graph.Img2Graph.Coloring.Coloring_Dataset import ColoringDataset

dataset_valid = ColoringDataset(
    root_path="Models/Any2Graph/Any2Graph/Img2Graph/Coloring/data",
    subset="small",
    split="valid",
)
dataset_test = ColoringDataset(
    root_path="Models/Any2Graph/Any2Graph/Img2Graph/Coloring/data",
    subset="small",
    split="test",
)
len(dataset_valid), len(dataset_test)

Loading Images...
...images loaded, it took 0.01 seconds
Loading Graphs...
...graphs loaded, it took 1.21
Loading Images...
...images loaded, it took 0.01 seconds
Loading Graphs...
...graphs loaded, it took 1.38


(10000, 10000)

Load a pretrained Any2Graph model.

Refer to the [repository](https://github.com/KrzakalaPaul/Any2Graph) in order to train one.

In [2]:
import torch
from Any2Graph.model import Any2Graph_Model
from ruamel.yaml import YAML
from Any2Graph.Img2Graph import Img2Graph

yaml = YAML()
config = yaml.load(open("ColoringModel/args.yaml", "r"))
device = "cuda"


weights = torch.load("ColoringModel/best_model", map_location="cpu")
task = Img2Graph(config)
model = Any2Graph_Model(task, config)

model.load_state_dict(weights)
model.to(device)
model.eval()
None

Output Shape of the CNN: torch.Size([64, 256, 8, 8])


Define the FGW distance to use.

In [12]:
from conformal.fgw import FGW

fgw = FGW(prior="FD", cost="adjacency")

## Conformal Prediction

Perform standard conformal prediction.

First, compute the distances between predicted graphs and ground truth graphs with respect to the FGW distance we use.

In [13]:
from conformal.any2graph import A2GGraph, sparse_to_graph, sparse_from_batch
from Any2Graph.graphs.custom_graphs_classes import BatchedContinuousGraphs
from torch import Tensor
from conformal.graph import Graph
from torch.utils.data import DataLoader
from tqdm import tqdm


def collate_fn(batch: list[tuple[Tensor, A2GGraph, int]]):
    images = torch.stack([sample[0] for sample in batch])
    graphs = [sample[1] for sample in batch]
    return images, graphs


@torch.no_grad()
def compute_distances(
    dataset: ColoringDataset, batch_size=64
) -> tuple[list[float], list[Graph]]:
    """Compute the distances from prediction to ground truth.
    Also returns the list of predicted graph for further comparison to other candidates.
    """

    distances: list[float] = []
    predictions: list[Graph] = []
    loader = DataLoader(dataset, batch_size, collate_fn=collate_fn)

    for images, graphs in tqdm(loader):
        out: BatchedContinuousGraphs = model(images.to(device), logits=False).to("cpu")
        preds = sparse_from_batch(out)

        for pred, truth in zip(preds, graphs):
            pred = Graph.from_a2g(sparse_to_graph(pred))
            distances.append(fgw(Graph.from_a2g(truth), pred))
            predictions.append(pred)

    return distances, predictions

In [14]:
distances_valid, preds_valid = compute_distances(dataset_valid)
distances_test, preds_test = compute_distances(dataset_test)

100%|██████████| 157/157 [00:07<00:00, 21.63it/s]


Compute the conformal threshold on the validation set.

In [15]:
from conformal.regression import FixedRegressor

fixed_regressor = FixedRegressor(target=0.9)
fixed_regressor.fit(distances_valid)

Now compute distances between prediction and all candidates in order to obtain the metrics.

First, compute candidate sets by grouping together graphs that have the same amount of nodes and the same amount of each color of nodes.

In [16]:
from conformal.any2graph import equivalence_classes

classes_test = equivalence_classes(dataset_test)

In [25]:
from conformal.any2graph import graph_class
from conformal.metrics import Metrics


def compute_metrics(
    regressor: FixedRegressor, dataset: ColoringDataset, predictions: list[Graph]
) -> Metrics:
    correct_coverage: list[bool] = []
    candidate_sizes: list[int] = []
    conformal_sizes: list[int] = []

    n = len(dataset)
    for i in tqdm(range(n), total=n):
        truth = Graph.from_a2g(dataset[i][1])
        pred = predictions[i]
        correct = fgw(truth, pred) <= regressor.threshold()
        candidates = classes_test[graph_class(dataset[i][1])]
        conformal_size = 0

        for j in candidates:
            if fgw(pred, Graph.from_a2g(dataset[j][1])) <= regressor.threshold():
                conformal_size += 1

        correct_coverage.append(correct)
        candidate_sizes.append(len(candidates))
        conformal_sizes.append(conformal_size)

    return Metrics(correct_coverage, candidate_sizes, conformal_sizes)


metrics_test = compute_metrics(fixed_regressor, dataset_test, preds_test)
metrics_test

100%|██████████| 10000/10000 [26:24<00:00,  6.31it/s] 


coverage: 89.5%
mean set size: 4
median set size: 1
mean reduction: 96.2%
median reduction: 98.9%
empty rate: 9.0%

## Score Conformalized Quantile Regression (SCQR)

SCQR on image embeddings.

In order to use entropies instead, you can compute the binary cross entropy of adjacency matrix and node feature predictions from the Any2Graph model.

First, compute image embeddings using the Any2Graph encoder.

In [9]:
from Any2Graph.Img2Graph.Img2Graph_Encoder import TroncatedResNet

encoder: TroncatedResNet = model.encoder  # type: ignore


def compute_img_embeds(dataset: ColoringDataset, batch_size=64) -> Tensor:
    def collate_fn(batch):
        return torch.stack([sample[0] for sample in batch])

    loader = DataLoader(dataset, batch_size, collate_fn=collate_fn)

    embeds_batchs = []

    for images in tqdm(loader):
        with torch.no_grad():
            x, _, _ = encoder(images.to(device))
        embeds_batchs.append(x.mean(dim=1).cpu())

    return torch.cat(embeds_batchs)

In [10]:
embeds_valid = compute_img_embeds(dataset_valid)
embeds_test = compute_img_embeds(dataset_test)

100%|██████████| 157/157 [00:01<00:00, 135.18it/s]


Train the quantile regression model with the embeddings.

In [18]:
from conformal.regression import EmbeddingRegressor

embedding_regressor = EmbeddingRegressor(target=0.9, embed_dim=embeds_valid.size(-1))
embedding_regressor.fit(distances_valid, embeds_valid)

Epoch 1 - Average loss: 0.0173
Epoch 2 - Average loss: 0.0104
Epoch 3 - Average loss: 0.0087
Epoch 4 - Average loss: 0.0074
Epoch 5 - Average loss: 0.0052
Epoch 6 - Average loss: 0.0042
Epoch 7 - Average loss: 0.0033
Epoch 8 - Average loss: 0.0033
Epoch 9 - Average loss: 0.0031
Epoch 10 - Average loss: 0.0027
Epoch 11 - Average loss: 0.0026
Epoch 12 - Average loss: 0.0023
Epoch 13 - Average loss: 0.0021
Epoch 14 - Average loss: 0.0021
Epoch 15 - Average loss: 0.0021
Epoch 16 - Average loss: 0.0020
Epoch 17 - Average loss: 0.0020
Epoch 18 - Average loss: 0.0020
Epoch 19 - Average loss: 0.0020


Finally, compute the metrics again.

In [19]:
def compute_metrics_scqr(
    regressor: EmbeddingRegressor,
    dataset: ColoringDataset,
    predictions: list[Graph],
    embeds: Tensor,
) -> Metrics:
    correct_coverage: list[bool] = []
    candidate_sizes: list[int] = []
    conformal_sizes: list[int] = []

    n = len(dataset)
    for i in tqdm(range(n), total=n):
        truth = Graph.from_a2g(dataset[i][1])
        pred = predictions[i]
        threshold = regressor.threshold(embeds[i])
        correct = fgw(truth, pred) <= threshold
        candidates = classes_test[graph_class(dataset[i][1])]
        conformal_size = 0

        for j in candidates:
            if fgw(pred, Graph.from_a2g(dataset[j][1])) <= threshold:
                conformal_size += 1

        correct_coverage.append(correct)
        candidate_sizes.append(len(candidates))
        conformal_sizes.append(conformal_size)

    return Metrics(correct_coverage, candidate_sizes, conformal_sizes)

In [20]:
metrics_test_scqr = compute_metrics_scqr(
    embedding_regressor, dataset_test, preds_test, embeds_test
)
metrics_test_scqr

100%|██████████| 10000/10000 [26:09<00:00,  6.37it/s] 


coverage: 93.0%
mean set size: 4
median set size: 1
mean reduction: 96.2%
median reduction: 98.9%
empty rate: 5.7%